# PatchTST Feature Importance Analysis

Dedicated notebook for asking: **of the 20 `hist_exog` covariates, which actually help `return_1h`'s forecast, and which are neutral or actively harmful?**

**Prerequisite**: `channel_attention=True` is required. With `channel_attention=False` (this project's Run 2/2a/2b/2c baseline), the other 20 channels are computed but structurally discarded — zero gradient, zero information reaches `return_1h` — so asking "which feature helps" has no meaningful answer under that config. This notebook trains its own `channel_attention=True` baseline (same structure as `docs/experiments.md` Run 3/4: channel-0-only loss, `patch_stride` fix, mean pooling) so the question is actually answerable.

**Methodology** (cheapest/most-trustworthy first):
1. **Training loss curve** — this project's other notebooks never logged loss during training, only post-hoc financial metrics, which can't distinguish "model didn't converge" from "target has no signal." Logged here.
2. **Grouped permutation importance** (primary) — shuffle each feature *group* across test windows, measure the increase in test MAE. Cheap (inference only, no retraining), and grouping avoids the collinearity trap (MACD variants, `vol_24h`/`vol_60h`/`bb_width`, OHLC are all highly correlated — shuffling one alone can look unimportant if a correlated feature substitutes for it).
3. **Individual permutation importance** (secondary) — same idea per-feature, for finer detail once a group looks interesting.
4. **Channel-attention weights** (qualitative only) — what the model *looks at*, not proof of causal contribution. High attention without permutation-importance backing should not be trusted alone.

**Caveats — read before trusting any specific result**:
- This is **one trained model, one seed**. Every other finding in this project that wasn't checked across multiple seeds turned out to be at least partly noise (Sharpe swinging by >1.0, `input_size` tuning winners flipping, directional-accuracy "wins" collapsing under a bias check). Treat anything here as a hypothesis to confirm with reruns, not a conclusion.
- Permutation importance measures *this model's* reliance on a feature, not whether the feature is inherently predictive — a feature could be useful but the model failed to learn to use it, or vice versa (overfit to noise in it).
- No transaction costs are modeled anywhere in this project's Sharpe/Max DD calculation — keep that in mind if any importance result is framed in terms of trading performance.

In [ ]:
# ── Colab Setup (skip automatically if running locally) ─────────────────────────────────────────
# This is an ongoing analysis tool, not a one-off experiment to be reproduced
# against a pinned historical commit — it always clones the latest `Model`
# branch rather than pinning TARGET_COMMIT (contrast with patchtst.ipynb /
# deepar.ipynb, which pin per docs/experiments.md's reproducibility workflow).
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', 'transformers', '-q'], check=True)

    REPO_URL = 'https://github.com/WoodyChang21/ECE1508_GenAI.git'
    if not os.path.exists('/content/ECE1508_GenAI'):
        subprocess.run(
            ['git', 'clone', '--branch', 'Model', REPO_URL, '/content/ECE1508_GenAI'],
            check=True,
        )
    subprocess.run(['git', '-C', '/content/ECE1508_GenAI', 'fetch', 'origin'], check=True)
    # reset --hard + clean -fd (not checkout): the /content clone persists across
    # cell reruns in one Colab session, so a prior run's local changes would make
    # a plain `checkout`/`pull` fail or silently keep stale files. This clone is
    # disposable — always discard and resync to the latest pushed commit.
    subprocess.run(['git', '-C', '/content/ECE1508_GenAI', 'reset', '--hard', 'origin/Model'], check=True)
    subprocess.run(['git', '-C', '/content/ECE1508_GenAI', 'clean', '-fd'], check=True)
    os.chdir('/content/ECE1508_GenAI/notebooks')

    missing = [
        f for f in ['train.parquet', 'val.parquet', 'test.parquet']
        if not os.path.exists(f'/content/ECE1508_GenAI/data/splits/{f}')
    ]
    if missing:
        raise FileNotFoundError('Missing data splits after clone: ' + ', '.join(missing))
    print('Data splits found from clone — no upload needed.')

    import torch
    if torch.cuda.is_available():
        print(f'\nGPU: {torch.cuda.get_device_name(0)}')
    else:
        print('\nNo GPU detected. Go to Runtime -> Change runtime type -> T4 GPU.')

print('Setup complete.')


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PatchTSTConfig, PatchTSTForPrediction
from scipy.stats import t as student_t
import matplotlib.pyplot as plt

from scripts.models.data_loader import HIST_EXOG_COLS
from scripts.models.metrics import compute_all

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

TARGET_COL = 'return_1h'
ALL_COLS   = [TARGET_COL] + HIST_EXOG_COLS   # 21 columns; return_1h is channel 0
INPUT_SIZE = 240   # fixed to the value that won most tuning passes across Runs 2/3/2a/2c —
                   # this notebook is about feature importance, not re-litigating lookback length
MAX_STEPS  = 1000  # matches this project's "final training" budget elsewhere
plt.rcParams['figure.figsize'] = (14, 4)

In [ ]:
# ── Helpers ─────────────────────────────────────────────────────

def patch_params(input_size: int):
    patch_len = max(4, min(16, input_size // 4))
    stride    = max(2, patch_len // 2)
    return patch_len, stride


class ZScoreScaler:
    """Fit on training data, transform any split with the same statistics."""
    def fit(self, data: np.ndarray):
        self.mean_ = data.mean(axis=0)
        self.std_  = data.std(axis=0) + 1e-8
        return self

    def transform(self, data: np.ndarray) -> np.ndarray:
        return (data - self.mean_) / self.std_

    def inverse_col0(self, z: np.ndarray) -> np.ndarray:
        return z * self.std_[0] + self.mean_[0]


class WindowDataset(Dataset):
    """
    past_values  : (context_length, 21)  — all channels, lookback window
    future_values: (1, 21)               — all channels at t+1 (channel 0 is the
                                            training target; see train_model)
    """
    def __init__(self, data: np.ndarray, context_length: int):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.ctx  = context_length

    def __len__(self):
        return len(self.data) - self.ctx

    def __getitem__(self, i):
        return {
            'past_values':   self.data[i : i + self.ctx],
            'future_values': self.data[i + self.ctx : i + self.ctx + 1],
        }


def build_model(input_size: int) -> PatchTSTForPrediction:
    patch_len, stride = patch_params(input_size)
    config = PatchTSTConfig(
        num_input_channels=len(ALL_COLS),
        context_length=input_size,
        prediction_length=1,
        patch_length=patch_len,
        patch_stride=stride,   # real PatchTSTConfig param name (see docs/experiments.md —
                               # `stride=` is silently absorbed as an unused kwarg by HF's
                               # PretrainedConfig and falls back to the library default)
        d_model=128,
        num_attention_heads=8,
        num_hidden_layers=3,
        dropout=0.2,
        head_dropout=0.0,
        channel_attention=True,   # required: the only config where the other 20 channels
                                  # can influence return_1h's forecast at all
        loss='nll',
        distribution_output='student_t',
    )
    return PatchTSTForPrediction(config)


def _distribution_params(model: PatchTSTForPrediction, past_values: torch.Tensor):
    """
    Bypass model.head.forward()'s final tuple-transpose step, which assumes
    prediction_length > 1 and raises IndexError with a distribution head at
    prediction_length=1 (verified against the transformers source: StudentTOutput's
    domain_map squeezes the size-1 forecast_len dim away entirely at prediction_length=1,
    but head.forward()'s later `.transpose(2, 1)` still assumes that dim exists).

    Uses mean pooling (pooling_type='mean', PatchTSTConfig's default) — same as
    docs/experiments.md Run 2/3/4, kept fixed here so this notebook isolates the
    feature-importance question rather than also varying pooling mode.
    """
    base_out = model.model(past_values=past_values)
    pooled   = base_out.last_hidden_state.mean(dim=2)             # mean-pool across patches: (bs, channels, d_model)
    pooled   = model.head.dropout(model.head.flatten(pooled))
    df, loc, scale = model.head.projection(pooled)                # ParameterProjection -> (bs, channels) each

    win_loc   = base_out.loc[:, 0, :]     # (bs, channels)
    win_scale = base_out.scale[:, 0, :]
    return df, loc, scale, win_loc, win_scale


def train_model(
    model: PatchTSTForPrediction,
    data: np.ndarray,
    context_length: int,
    max_steps: int,
    batch_size: int = 64,
    log_every: int = 10,
):
    """
    Same training loop as this project's other PatchTST notebooks, plus loss
    logging (recorded every `log_every` steps) — this project's other notebooks
    never logged training loss, only post-hoc financial metrics, which can't
    distinguish "model didn't converge" from "target has no signal."

    Returns (model, loss_history) where loss_history is a list of (step, loss) pairs.
    """
    dataset = WindowDataset(data, context_length)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    opt     = torch.optim.Adam(model.parameters(), lr=1e-4)
    model.to(DEVICE).train()
    loss_history = []
    step = 0
    while step < max_steps:
        for batch in loader:
            pv = batch['past_values'].to(DEVICE)
            fv = batch['future_values'].to(DEVICE)[:, 0, :]   # (bs, 21) — squeeze prediction_length=1 axis

            df, loc, scale, win_loc, win_scale = _distribution_params(model, pv)

            # Loss restricted to channel 0 (return_1h) only, matching DeepAR and this
            # project's Run 2 fix (loss averaged across all 21 channels was diagnosed
            # as causing a persistence/lag-echo strategy — see docs/experiments.md Run 1/2).
            target0   = fv[:, 0]
            df0, loc0, scale0     = df[:, 0], loc[:, 0], scale[:, 0]
            win_loc0, win_scale0  = win_loc[:, 0], win_scale[:, 0]

            standardized_target = (target0 - win_loc0) / win_scale0
            base_dist = torch.distributions.StudentT(df=df0, loc=loc0, scale=scale0)
            log_prob  = base_dist.log_prob(standardized_target) - torch.log(win_scale0)
            loss = -log_prob.mean()

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step += 1

            if step == 1 or step % log_every == 0:
                loss_history.append((step, float(loss.item())))
            if step >= max_steps:
                break
    return model, loss_history


def build_windows(full_data: np.ndarray, context_length: int, start_idx: int) -> np.ndarray:
    """(n, context_length, num_channels) windows, one per test-set prediction origin."""
    return np.stack([
        full_data[i - context_length : i]
        for i in range(start_idx, len(full_data))
    ])


def predict_from_windows(model: PatchTSTForPrediction, windows: np.ndarray, batch_size: int = 512):
    """
    Same analytic-forecast logic as predict_distribution below, but takes
    pre-built windows directly — lets permutation-importance code feed in
    windows with specific channels shuffled, without rebuilding from scratch.
    """
    model.eval()
    means, lo_80, hi_80, lo_90, hi_90 = [], [], [], [], []
    loc_raw_list, win_loc_list, win_scale_list = [], [], []
    with torch.no_grad():
        for i in range(0, len(windows), batch_size):
            x = torch.tensor(windows[i : i + batch_size], dtype=torch.float32).to(DEVICE)
            df, loc, scale, win_loc, win_scale = _distribution_params(model, x)

            df0, loc0, scale0 = df[:, 0].cpu().numpy(), loc[:, 0].cpu().numpy(), scale[:, 0].cpu().numpy()
            wloc0, wscale0    = win_loc[:, 0].cpu().numpy(), win_scale[:, 0].cpu().numpy()

            mean_z = loc0 * wscale0 + wloc0
            q = {lvl: student_t.ppf(lvl, df0, loc=loc0, scale=scale0) * wscale0 + wloc0
                 for lvl in [0.05, 0.10, 0.90, 0.95]}

            means.append(mean_z)
            lo_80.append(q[0.10]); hi_80.append(q[0.90])
            lo_90.append(q[0.05]); hi_90.append(q[0.95])
            loc_raw_list.append(loc0); win_loc_list.append(wloc0); win_scale_list.append(wscale0)

    return (np.concatenate(means), np.concatenate(lo_80), np.concatenate(hi_80),
            np.concatenate(lo_90), np.concatenate(hi_90),
            np.concatenate(loc_raw_list), np.concatenate(win_loc_list), np.concatenate(win_scale_list))


def predict_distribution(model, full_data, context_length, start_idx, batch_size=512):
    windows = build_windows(full_data, context_length, start_idx)
    return predict_from_windows(model, windows, batch_size)


def permute_channels(windows: np.ndarray, channel_indices: list, rng: np.random.Generator) -> np.ndarray:
    """
    Shuffle the given channel indices across the sample (window) dimension,
    using the SAME permutation for every channel in the group — this preserves
    each group's internal joint structure while destroying its alignment with
    the target and with every other (non-permuted) channel.
    """
    out  = windows.copy()
    perm = rng.permutation(windows.shape[0])
    for idx in channel_indices:
        out[:, :, idx] = windows[perm, :, idx]
    return out


def eval_mae_on_windows(model, windows: np.ndarray, z_true: np.ndarray, batch_size: int = 512) -> float:
    y_pred_z, *_ = predict_from_windows(model, windows, batch_size)
    return float(np.mean(np.abs(y_pred_z - z_true)))

In [ ]:
# ── Load and normalise data ─────────────────────────────────────────
train_raw = pd.read_parquet('../data/splits/train.parquet')
val_raw   = pd.read_parquet('../data/splits/val.parquet')
test_raw  = pd.read_parquet('../data/splits/test.parquet')

for df in [train_raw, val_raw, test_raw]:
    df['is_first_bar'] = df['is_first_bar'].astype(float)

train_mat = train_raw[ALL_COLS].values.astype(np.float32)
val_mat   = val_raw[ALL_COLS].values.astype(np.float32)
test_mat  = test_raw[ALL_COLS].values.astype(np.float32)

scaler     = ZScoreScaler().fit(train_mat)
train_norm = scaler.transform(train_mat)
val_norm   = scaler.transform(val_mat)
test_norm  = scaler.transform(test_mat)

trainval_norm = np.concatenate([train_norm, val_norm], axis=0)
full_norm     = np.concatenate([train_norm, val_norm, test_norm], axis=0)

n_train, n_val, n_test = len(train_norm), len(val_norm), len(test_norm)

print(f'Train: {n_train:,}  Val: {n_val:,}  Test: {n_test:,}')
print(f'Input channels ({len(ALL_COLS)}): {ALL_COLS}')

In [ ]:
# ── Train the baseline (all-features) model ──────────────────────────────
model, loss_history = train_model(
    build_model(INPUT_SIZE), trainval_norm, INPUT_SIZE, max_steps=MAX_STEPS,
)
print(f'Trained on {len(trainval_norm):,} rows, input_size={INPUT_SIZE}, '
      f'{len(loss_history)} logged loss points over {MAX_STEPS} steps')

In [ ]:
# ── Training loss curve ────────────────────────────────────────────────
# Read this before trusting anything downstream: if the loss is still clearly
# decreasing at the final step, max_steps is likely too low for this config
# (undertrained). If it's flat/plateaued well before the end, more steps won't
# help and any "no signal" conclusion is on firmer ground.
steps, losses = zip(*loss_history)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses)
ax.set_xlabel('training step')
ax.set_ylabel('NLL loss (channel-0 only)')
ax.set_title('Training loss curve')
plt.tight_layout()
plt.show()

In [ ]:
# ── Baseline test metrics (all features) ──────────────────────────────
test_start_idx = n_train + n_val
baseline_windows = build_windows(full_norm, INPUT_SIZE, test_start_idx)
z_test_true = test_norm[:, 0]

(y_pred_z, lo_80_z, hi_80_z, lo_90_z, hi_90_z,
 loc_raw, win_loc, win_scale) = predict_from_windows(model, baseline_windows)

y_true = scaler.inverse_col0(z_test_true)
y_pred = scaler.inverse_col0(y_pred_z)
lo_80  = scaler.inverse_col0(lo_80_z)
hi_80  = scaler.inverse_col0(hi_80_z)
lo_90  = scaler.inverse_col0(lo_90_z)
hi_90  = scaler.inverse_col0(hi_90_z)

results = compute_all(y_true, y_pred, lo_80, hi_80, lo_90, hi_90)
baseline_mae = float(np.mean(np.abs(y_pred_z - z_test_true)))   # in normalised space, used by permutation tests below

print('=== Baseline (all features) PatchTST test results ===')
print(f"  RMSE              : {results['rmse']:.6f}")
print(f"  MAE               : {results['mae']:.6f}")
print(f"  Directional Acc   : {results['dir_acc']:.4f}")
print(f"  Coverage 80%      : {results['coverage_80']:.4f}  (target: 0.80)")
print(f"  Coverage 90%      : {results['coverage_90']:.4f}  (target: 0.90)")
print(f"  Sharpe Ratio      : {results['sharpe']:.4f}")
print(f"  Max Drawdown      : {results['max_drawdown']:.6f}")
print(f"  (normalised-space MAE used for permutation importance below: {baseline_mae:.6f})")

y_lag1_norm  = np.roll(z_test_true, 1)
corr_loc_raw = np.corrcoef(loc_raw[1:], y_lag1_norm[1:])[0, 1]
print(f"\n  corr(loc_raw, y[t-1]) : {corr_loc_raw:.4f}   <- lag-echo diagnostic, for reference against docs/experiments.md")

## Feature importance

Features are grouped to avoid the collinearity trap: several of these are near-duplicate transformations of the same underlying price series (MACD/MACD-signal/MACD-diff; `vol_24h`/`vol_60h`/`bb_width`; the four OHLC columns), so shuffling one alone can look falsely unimportant if a correlated feature in the same group substitutes for it.

In [ ]:
FEATURE_GROUPS = {
    'return_1h (own history, positive control)': ['return_1h'],   # channel 0 itself — sanity check: should
                                                                     # show large importance since even a
                                                                     # channel-independent model relies heavily
                                                                     # on its own patch history
    'price (OHLC)':          ['open', 'high', 'low', 'close'],
    'lagged returns':        ['return_4h', 'return_24h'],
    'volatility/bollinger':  ['vol_24h', 'vol_60h', 'bb_width', 'bb_upper', 'bb_lower'],
    'momentum':              ['rsi_14', 'macd', 'macd_signal', 'macd_diff'],
    'volume':                ['volume', 'volume_ratio'],
    'market regime (VIX)':   ['vix_log', 'vix_change_1h'],
    'calendar':              ['is_first_bar'],
}

# sanity check: every non-target column accounted for exactly once
grouped_cols = sorted(c for g in FEATURE_GROUPS.values() for c in g if c != 'return_1h')
assert grouped_cols == sorted(HIST_EXOG_COLS), f'group membership mismatch: {set(HIST_EXOG_COLS) ^ set(grouped_cols)}'
print(f'{len(FEATURE_GROUPS)} groups covering all {len(ALL_COLS)} channels (incl. the return_1h control group).')

In [ ]:
# ── Grouped permutation importance ──────────────────────────────────────
# For each group: shuffle it across test windows N_REPEATS times (averaging out
# the permutation draw's own randomness) and record the increase in test MAE
# vs. the unpermuted baseline. Positive = model relies on it (helpful to remove
# would hurt). Near-zero or negative = not used, or actively counterproductive.
N_REPEATS = 5
rng = np.random.default_rng(0)

grouped_importance = {}
for group_name, cols in FEATURE_GROUPS.items():
    channel_indices = [ALL_COLS.index(c) for c in cols]
    deltas = []
    for _ in range(N_REPEATS):
        permuted_windows = permute_channels(baseline_windows, channel_indices, rng)
        permuted_mae = eval_mae_on_windows(model, permuted_windows, z_test_true)
        deltas.append(permuted_mae - baseline_mae)
    grouped_importance[group_name] = (float(np.mean(deltas)), float(np.std(deltas)))
    print(f'  {group_name:45s} ΔMAE = {np.mean(deltas):+.6f} (±{np.std(deltas):.6f} over {N_REPEATS} shuffles)')

sorted_groups = sorted(grouped_importance.items(), key=lambda kv: kv[1][0], reverse=True)
fig, ax = plt.subplots(figsize=(10, 5))
names = [k for k, _ in sorted_groups]
means = [v[0] for _, v in sorted_groups]
stds  = [v[1] for _, v in sorted_groups]
ax.barh(names, means, xerr=stds, color='steelblue')
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Δ test MAE when group is shuffled (higher = more relied upon)')
ax.set_title('Grouped permutation importance')
plt.tight_layout()
plt.show()

In [ ]:
# ── Individual permutation importance (all 20 features + return_1h) ────────
# Finer-grained follow-up to the grouped result above — cheap since it's
# inference-only, so run it for every feature rather than only "useful" groups.
individual_importance = {}
for col in ALL_COLS:
    idx = ALL_COLS.index(col)
    deltas = []
    for _ in range(N_REPEATS):
        permuted_windows = permute_channels(baseline_windows, [idx], rng)
        permuted_mae = eval_mae_on_windows(model, permuted_windows, z_test_true)
        deltas.append(permuted_mae - baseline_mae)
    individual_importance[col] = (float(np.mean(deltas)), float(np.std(deltas)))

sorted_features = sorted(individual_importance.items(), key=lambda kv: kv[1][0], reverse=True)
fig, ax = plt.subplots(figsize=(10, 7))
names = [k for k, _ in sorted_features]
means = [v[0] for _, v in sorted_features]
stds  = [v[1] for _, v in sorted_features]
ax.barh(names, means, xerr=stds, color='darkorange')
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Δ test MAE when feature is shuffled (higher = more relied upon)')
ax.set_title('Individual permutation importance')
plt.tight_layout()
plt.show()

for col, (mean, std) in sorted_features:
    print(f'  {col:16s} ΔMAE = {mean:+.6f} (±{std:.6f})')

## Channel-attention weights (qualitative only — not a substitute for the permutation results above)

**Important library detail found while building this**: `PatchTSTEncoder.forward()`'s own `output_attentions=True` path only exposes the *time*-attention weights (sublayer 1). It silently discards the *channel*-attention weights (sublayer 2, only computed when `channel_attention=True`) — confirmed by reading `PatchTSTEncoder.forward`'s source: it only appends `layer_outputs[1]` to `all_attentions`, never `layer_outputs[2]`. So a naive `model(..., output_attentions=True)` call would silently hand you the wrong attention matrix. The cell below bypasses this by replicating the encoder's forward pass layer-by-layer and capturing `layer_outputs[2]` directly.

Treat this purely as "what does the model look at" — high attention to a channel does not by itself prove that channel improves the forecast (that's what the permutation results above are for).

In [ ]:
# ── Extract channel-attention weights (bypasses the encoder's own output_attentions path) ──
def get_channel_attention_from_return1h(model, past_values: torch.Tensor):
    """
    Returns a list (one entry per transformer layer) of channel-attention weight
    tensors, each shaped (batch_size * num_patches, num_heads, num_channels, num_channels)
    per the encoder-layer source (sublayer 2 reshapes to
    (batch_size * sequence_length, num_input_channels, d_model) before attention,
    where "sequence_length" here is num_patches).

    NOTE: this replicates PatchTSTModel/PatchTSTEncoder's internal forward steps
    manually (scaler -> patchifier -> embedder -> positional_encoder -> layers),
    since there's no public API to get sublayer-2 attention. Verify the printed
    shape below matches (*, num_heads, num_channels, num_channels) before trusting
    the reduction in the next cell — if a future transformers version changes this
    internal structure, this will need updating.
    """
    encoder = model.model.encoder
    with torch.no_grad():
        observed = torch.ones_like(past_values)
        scaled, _, _ = model.model.scaler(past_values, observed)
        patched = model.model.patchifier(scaled)
        hidden_state = encoder.positional_encoder(encoder.embedder(patched))

        channel_attns = []
        for layer in encoder.layers:
            layer_outputs = layer(hidden_state=hidden_state, output_attentions=True)
            hidden_state = layer_outputs[0]
            if layer.channel_attention:
                channel_attns.append(layer_outputs[2].detach().cpu())
    return channel_attns


sample_windows = baseline_windows[:512]   # subsample for speed; attention doesn't need the full test set
sample_x = torch.tensor(sample_windows, dtype=torch.float32).to(DEVICE)
channel_attns = get_channel_attention_from_return1h(model, sample_x)

print(f'{len(channel_attns)} layer(s) with channel attention. Shape of layer 0: {tuple(channel_attns[0].shape)}')
print('Expected: (batch_size * num_patches, num_heads, num_channels, num_channels) '
      f'-> last two dims should both equal {len(ALL_COLS)}. Verify before trusting the plot below.')

In [ ]:
# Average over batch*num_patches and heads, then over layers -> for each layer,
# row 0 = how much channel 0 (return_1h, the query) attends to each channel (key).
assert channel_attns[0].shape[-1] == len(ALL_COLS) and channel_attns[0].shape[-2] == len(ALL_COLS), (
    'channel-attention shape does not match (*, heads, channels, channels) as expected — '
    'inspect the printed shape above before trusting this reduction.'
)

per_layer_attn_to_channel0_query = [w.mean(dim=(0, 1))[0, :].numpy() for w in channel_attns]   # each: (num_channels,)
avg_attn = np.mean(per_layer_attn_to_channel0_query, axis=0)   # averaged across layers: (num_channels,)

order = np.argsort(avg_attn)[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh([ALL_COLS[i] for i in order], avg_attn[order], color='seagreen')
ax.set_xlabel('mean channel-attention weight (return_1h query -> each channel key)')
ax.set_title('Channel attention paid by return_1h to each channel (qualitative only)')
plt.tight_layout()
plt.show()

for i in order:
    print(f'  {ALL_COLS[i]:16s} attn = {avg_attn[i]:.4f}')

## Caveats and recommended next steps

- **This is one seed, one trained model.** Before trusting any specific feature/group's importance ranking, rerun this notebook with a different seed (and ideally 3-5 seeds) and check whether the ranking holds — this project's own log has repeatedly found single-run results (Sharpe, directional accuracy, `input_size` tuning winners) to be substantially noise.
- **Grouped/individual permutation importance measures this model's reliance on a feature**, not whether the feature is inherently predictive. A feature could be genuinely useful but the model failed to learn to use it (undertraining, optimization difficulty), or the model could be relying on a feature that's really just noise it overfit to — the training loss curve above is the first place to check if results here look surprising.
- **Channel-attention weights are qualitative only.** A channel can receive high attention without the permutation test showing it matters (attention isn't guaranteed to equal causal contribution), and vice versa.
- **If a group/feature looks consistently important across a few seeds**, the next step up in rigor is full leave-one-out retraining (drop that feature from `ALL_COLS`/`HIST_EXOG_COLS`, retrain from scratch, compare test MAE) — expensive, so reserve it for whichever candidates survive the cheaper permutation-based screening here.
- **No transaction costs are modeled** in this project's Sharpe/Max Drawdown calculation (`scripts/models/metrics.py`) — if any feature's importance is framed in terms of trading performance rather than point-forecast error, keep that caveat in mind.